<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 7: Ensemble Modeller (RF, XGBoost, LightGBM)

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 7 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_ensemble_modeller.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_ensemble_modeller.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta07_ensemble_modeller.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/07/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Ensemble Learning: Bagging vs Boosting
> - Random Forest, XGBoost, LightGBM
> - Feature Importance analizi

# Hafta 7 — Topluluk (Ensemble) Modelleri

Bu defterde topluluk öğrenme yöntemlerini öğrenecek ve karşılaştıracağız.

## İçindekiler
1. Bagging vs Boosting Kavramları
2. Random Forest (Rastgele Orman)
3. XGBoost
4. LightGBM
5. Özellik Önem Grafikleri
6. Model Doğruluk Karşılaştırması

## 1. Bagging vs Boosting

**Topluluk (Ensemble) öğrenme**, birden fazla modeli birleştirerek daha güçlü bir model elde etme yöntemidir.

### Bagging (Bootstrap Aggregating)
- Veriden **rastgele örneklemler** alınarak birden fazla model eğitilir
- Modeller **bağımsız** ve **paralel** çalışır
- Sonuçlar **oylama** (sınıflandırma) veya **ortalama** (regresyon) ile birleştirilir
- **Varyansı düşürür**, aşırı öğrenmeyi azaltır
- Örnek: **Random Forest**

### Boosting
- Modeller **sıralı** olarak eğitilir
- Her yeni model, önceki modelin **hatalarına odaklanır**
- Zayıf öğreniciler birleştirilerek güçlü bir model oluşturulur
- **Yanlılığı (bias) düşürür**
- Örnekler: **XGBoost**, **LightGBM**, **AdaBoost**, **CatBoost**

| Özellik | Bagging | Boosting |
|---------|---------|----------|
| Eğitim şekli | Paralel | Sıralı |
| Odak | Varyans azaltma | Yanlılık azaltma |
| Aşırı öğrenme riski | Düşük | Orta-Yüksek |
| Hız | Hızlı | Görece yavaş |
| Örnek algoritma | Random Forest | XGBoost, LightGBM |

In [ ]:
# Gerekli paketleri kur
!pip install -q xgboost lightgbm

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `lightgbm` | Hızlı gradient boosting framework |
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `warnings` | Uyarı mesajlarını yönetme |
| `xgboost` | Gradient Boosting tabanlı güçlü sınıflandırma/regresyon |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Tüm kütüphaneler başarıyla yüklendi!")

### Veri Setinin Yüklenmesi

Aşağıdaki kodda veri setini yüklüyoruz ve temel bilgilerine (boyut, sütunlar, ilk satırlar) bakıyoruz. Bu adım her veri bilimi projesinin başlangıcıdır.

In [ ]:
# Iris veri setini yükle
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# Eğitim-test ayırma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")
print(f"Sınıflar:    {list(iris.target_names)}")

## 2. Random Forest (Rastgele Orman)

Random Forest, birden fazla karar ağacını birleştiren bir **bagging** yöntemidir. Her ağaç verinin rastgele bir alt kümesinde eğitilir.

In [ ]:
# Random Forest modeli
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Random Forest Doğruluğu: {rf_acc:.4f}")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, rf_pred, target_names=iris.target_names))

## 3. XGBoost

**XGBoost (Extreme Gradient Boosting)**, gradient boosting algoritmasının optimize edilmiş bir versiyonudur. Kaggle yarışmalarında en çok kullanılan algoritmalardan biridir.

In [ ]:
# XGBoost modeli
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Doğruluğu: {xgb_acc:.4f}")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, xgb_pred, target_names=iris.target_names))

## 4. LightGBM

**LightGBM**, Microsoft tarafından geliştirilen hızlı ve verimli bir gradient boosting kütüphanesidir. Büyük veri setlerinde XGBoost'tan daha hızlıdır.

In [ ]:
# LightGBM modeli
lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_acc = accuracy_score(y_test, lgbm_pred)

print(f"LightGBM Doğruluğu: {lgbm_acc:.4f}")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, lgbm_pred, target_names=iris.target_names))

## 5. Özellik Önem Grafikleri

Her modelin hangi özelliklere ne kadar önem verdiğini görselleştirelim.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

models_imp = {
    'Random Forest': rf_model.feature_importances_,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgbm_model.feature_importances_
}

colors_list = ['#2196F3', '#4CAF50', '#FF9800']

for ax, (name, importances), color in zip(axes, models_imp.items(), colors_list):
    imp_df = pd.DataFrame({
        'Özellik': iris.feature_names,
        'Önem': importances
    }).sort_values('Önem', ascending=True)
    
    ax.barh(imp_df['Özellik'], imp_df['Önem'], color=color, edgecolor='white')
    ax.set_title(f'{name}\nÖzellik Önem Değerleri')
    ax.set_xlabel('Önem')

plt.suptitle('Özellik Önem Karşılaştırması', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Model Doğruluk Karşılaştırması

### Çapraz Doğrulama

Modelin güvenilirliğini çapraz doğrulama ile ölçüyoruz. Veri k parçaya bölünür ve her turda farklı bir parça test seti olarak kullanılır. Bu yöntem tek bir bölünmeye bağımlılığı ortadan kaldırır.

In [ ]:
# Çapraz doğrulama ile karşılaştırma
cv_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, 
                             random_state=42, eval_metric='mlogloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, 
                               random_state=42, verbose=-1)
}

cv_results = {}
print("5 Katlı Çapraz Doğrulama Sonuçları")
print("=" * 55)

for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:20s} | Ortalama: {scores.mean():.4f} ± {scores.std():.4f}")

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Doğruluk çubuk grafiği
model_names = list(cv_results.keys())
mean_scores = [cv_results[n].mean() for n in model_names]
std_scores = [cv_results[n].std() for n in model_names]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, mean_scores, yerr=std_scores, capsize=8,
               color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='white',
               width=0.5, alpha=0.9)

for bar, score in zip(bars, mean_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', fontsize=13, fontweight='bold')

plt.ylabel('Doğruluk (Accuracy)')
plt.title('Topluluk Modelleri — Çapraz Doğrulama Doğruluğu')
plt.ylim(0.85, 1.02)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Kutu Grafiği (Box Plot)

Kutu grafiği verinin dağılımını, medyanını, çeyrekliklerini ve uç değerlerini (outlier) gösterir. Gruplar arası karşılaştırma için idealdir.

In [ ]:
# Kutu grafiği
plt.figure(figsize=(10, 6))
bp = plt.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
                 widths=0.4)

for patch, color in zip(bp['boxes'], ['#2196F3', '#4CAF50', '#FF9800']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.ylabel('Doğruluk (Accuracy)')
plt.title('Çapraz Doğrulama — Kutu Grafiği Karşılaştırması')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Özet

Bu defterde öğrendiklerimiz:

1. **Bagging**: Paralel model eğitimi, varyans azaltma (Random Forest)
2. **Boosting**: Sıralı model eğitimi, yanlılık azaltma (XGBoost, LightGBM)
3. **Random Forest**: Güçlü, aşırı öğrenmeye dayanıklı, yorumlanabilir
4. **XGBoost**: Kaggle favorisi, yüksek performans
5. **LightGBM**: Hızlı, bellek verimli, büyük veri setleri için ideal
6. **Özellik önemi**: Her model hangi özelliklere ne kadar ağırlık veriyor

### Sonraki Adım
Bu topluluk modellerini gerçek dünya senaryolarında (müşteri terk analizi) uygulayacağız!

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>